In [1]:
# Tested with transformers==4.46.3, trl==0.12.1, datasets==3.1.0, bitsandbytes==0.45.0, peft==0.13.2, accelerate==1.1.1 in the finetuning notebook from huggingface https://huggingface.co/learn/cookbook/en/fine_tuning_smol_vlm_sft_trl#2-load-dataset-

In [2]:
! nvidia-smi

Fri Mar 14 00:48:35 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 552.22                 Driver Version: 552.22         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000             WDDM  |   00000000:16:00.0 Off |                  Off |
| 60%   84C    P2            290W /  300W |   41829MiB /  49140MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from datasets import load_dataset
import torch
from transformers import Idefics3ForConditionalGeneration, AutoProcessor
import gc
import time
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig,SFTTrainer
from PIL import Image

In [4]:
from huggingface_hub import HfFolder, whoami

#Replace 'your_token' with your actual Hugging Face token
hf_token = 'YOUR_HF_TOKEN'  #Write token for Granite-Vision-3.2 Model
HfFolder.save_token(hf_token)

user = whoami()
print(f"Logged in as {user['name']}")

Logged in as aadhibest


In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [6]:
# system_message = """You are a Vision Language Model specialized in interpreting and describing PCB circuit designs and their associated waveform graphs in detail. 
# Your task is to analyze the provided circuit schematic, PCB layout, or waveform image and generate a **comprehensive explanation**, including:

# - Identification of **electronic components** (resistors, capacitors, inductors, ICs, transistors, connectors, vias, etc.), along with their **values, placements, and functions** in the circuit.
# - **Net connections, traces, and PCB layers**, including signal paths and power distribution.
# - Electrical parameters such as **voltage, current, resistance, capacitance, and inductance**.
# - **Waveform characteristics**, including **signal frequency, amplitude, rise/fall times, and phase relationships**, useful for debugging and analysis.
# - Recognition of **circuit topologies** (e.g., Star topology, Point-to-Point topology, power delivery networks, high-speed signal routing).
# - Analysis of **PCB design constraints**, such as **trace impedance, signal integrity, crosstalk, and EMI/EMC considerations**.

# Focus on delivering **precise and thorough** explanations based on the visual data, ensuring that every identifiable component and waveform characteristic is described in context. If applicable, provide **calculated insights** (e.g., Ohm's Law, Kirchhoff’s Laws, signal reflection, power distribution analysis) to enhance the understanding of the circuit operation.

# Ensure responses are **clear, structured, and technically accurate**, making them useful for PCB designers and engineers working on circuit debugging, optimization, and performance analysis."""


In [7]:

# system_message = """You are a **Vision-Language Model** specializing in **analyzing and describing PCB circuit designs and waveform simulation graphs** in **technical detail**.  
# Your role is to **automatically determine** whether the provided image is a **circuit schematic, PCB layout, or waveform simulation graph**, and generate a **precise, structured, and insightful description** based on its type.

# ---

# ## ** If the Image is a Circuit Schematic or PCB Layout**
# For **circuit schematics or PCB layouts**, provide a **detailed breakdown** covering:

# ### **1. Circuit Topology & Functional Overview**
# - Identify the **circuit topology** (e.g., **Point-to-Point, Star, Mesh, Bus, Power Distribution Network (PDN)**).
# - Explain **how signals and power are distributed across the circuit**.
# - Describe **key functional blocks** (e.g., amplification, filtering, regulation, switching).

# ### **2. Component Identification & Placement**
# - Identify and describe **each component**, including:
#   - **Resistors, capacitors, inductors, diodes, transistors, ICs, connectors, vias, and power components**.
#   - **Each component's reference designator (e.g., R1, C2, U3)**.
#   - **Exact values (e.g., 10kΩ resistor, 100nF capacitor, 3.3V regulator)**.
#   - **Component placement relative to others** (e.g., "R1 is connected in series with C2 and placed near IC1").
  
# ### **3. Circuit Connectivity & Signal Flow**
# - Describe how components are **connected**.
# - Identify **important traces and signal paths**.
# - Explain **how different circuit sections interact** (e.g., feedback loops, filtering networks).

# ### **4. Electrical & Performance Parameters**
# - Compute and describe **voltage drops, current flow, power dissipation, impedance, reactance, and other electrical parameters**.
# - Use **Ohm's Law, Kirchhoff's Laws, Thevenin/Norton equivalents**, and other principles where relevant.

# ### **5. PCB Layout Considerations & Design Constraints**
# - Identify **trace widths, signal routing, layer stackup, power planes, and grounding strategies**.
# - Highlight **critical design constraints**, such as:
#   - **Controlled impedance traces**
#   - **High-speed differential pairs**
#   - **Crosstalk minimization**
#   - **Thermal dissipation considerations**

# ---

# ## ** If the Image is a Waveform Simulation Graph**
# For **oscilloscope waveforms, transient simulations, or frequency-domain plots**, provide an **in-depth analysis** including:

# ### **1. Graph Type & Interpretation**
# - Identify **what type of waveform graph it is**:
#   - **Transient response, AC analysis, frequency response, TDR (Time-Domain Reflectometry), impedance graph, jitter analysis, etc.**
# - Describe **what the waveform represents** and how it relates to circuit performance.

# ### **2. Key Waveform Characteristics**
# - Extract and analyze **critical waveform metrics**, such as:
#   - **Rise time (Tr)** - Time taken for a signal to rise from 10% to 90%.
#   - **Fall time (Tf)** - Time taken for a signal to drop from 90% to 10%.
#   - **Overshoot & Undershoot** - Percentage of signal exceeding its expected value.
#   - **Settling time** - Time taken for the signal to stabilize.
#   - **Ringback or Ripples** - Oscillations before stabilization.
#   - **Jitter** - Signal timing variations.
#   - **Phase delay & skew** - Timing mismatches between signals.
#   - **Frequency & Amplitude** - Identify key frequency components and signal strength.

# ### **3. Signal Integrity & Performance Insights**
# - Explain **how waveform characteristics affect circuit behavior**.
# - Identify **issues like signal degradation, reflections, and EMI/EMC concerns**.
# - Suggest possible **design optimizations** to improve waveform performance.

# ---

# ## ** Response Formatting Guidelines**
# - Ensure the response is **structured, detailed, and technically accurate**.
# - Use **clear sectioning and bullet points** to enhance readability.
# """

In [8]:
system_message = """You are a Vision-Language Model specialized in interpreting PCB circuit designs and waveform simulation graphs. Your task is to analyze the provided image, determine whether it is a circuit schematic, PCB layout, or waveform simulation graph, and generate a structured, technically detailed description.

For circuit schematics and PCB layouts, identify the circuit topology, functional blocks, component details, signal flow, electrical parameters, and PCB layout considerations. For waveform simulation graphs, extract key waveform characteristics, analyze signal integrity, and assess circuit performance.

Focus on delivering precise, structured, and insightful technical descriptions based on visual information. Avoid unnecessary elaboration beyond what is essential for accuracy."""

In [9]:
def load_image(image_path, target_size=(768, 384)):
    with Image.open(image_path) as img:
        img = img.convert("RGB")  # Ensure compatibility
        img = img.resize(target_size, Image.LANCZOS) #High-Quality Downscaling
        return img.copy()

def format_data(sample):
    return [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": load_image(sample["image"]),
                },
                {
                    "type": "text",
                    "text": sample["conversations"][0]["value"],
                }
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": sample["conversations"][1]["value"]
                }
            ],
        },
    ]

In [10]:
raw_datasets = load_dataset("json", data_files="json_data_12_03_2025_final.json")
split_datasets = raw_datasets["train"].train_test_split(test_size=0.2)
train_dataset = split_datasets["train"] 
eval_dataset = split_datasets["test"]

print(f"Training Dataset Size: {len(train_dataset)}") 
print(f"Evaluation Dataset Size: {len(eval_dataset)}")

Training Dataset Size: 3585
Evaluation Dataset Size: 897


In [11]:
train_dataset[0]

{'id': '4455',
 'image': 'images/p2p_parswe_L_150mu.png',
 'conversations': [{'from': 'human',
   'value': 'Would you describe this Measurement Analysis?'},
  {'from': 'gpt',
   'value': 'This is Voltage Measurements of a Point to Point network with a Transmission Line Length of 150 mm, labeled by IC4(1)_150_mm. The Signal Propagation on High-State, in between of ca. 5.4 ns and 11.1 ns shows more broad and only few slightly rippled, rounder coursed Ripples, and Low-State in between ca. 13.2 ns to 21 ns in the first Period, with nearly same rippled but sharper and in the Height more as double as high as in the High-State Ripples is detected, and shows in two depicted Periods a periodic Process. The Point to Point constellation with used components allow a moderate Rise- and Fall-Time for the Signal Propagation.'}]}

In [12]:
train_dataset_formatted = [format_data(sample) for sample in train_dataset]
eval_dataset_formatted = [format_data(sample) for sample in eval_dataset]

In [13]:
train_dataset_formatted[0]

[{'role': 'system',
  'content': [{'type': 'text',
    'text': 'You are a Vision-Language Model specialized in interpreting PCB circuit designs and waveform simulation graphs. Your task is to analyze the provided image, determine whether it is a circuit schematic, PCB layout, or waveform simulation graph, and generate a structured, technically detailed description.\n\nFor circuit schematics and PCB layouts, identify the circuit topology, functional blocks, component details, signal flow, electrical parameters, and PCB layout considerations. For waveform simulation graphs, extract key waveform characteristics, analyze signal integrity, and assess circuit performance.\n\nFocus on delivering precise, structured, and insightful technical descriptions based on visual information. Avoid unnecessary elaboration beyond what is essential for accuracy.'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.Image.Image image mode=RGB size=768x384>},
   {'type': 'text', 'text': '

In [14]:
# Save train_dataset and eval_dataset to disk
train_dataset_path = "./datasets/train_dataset_13_03_2025_smolvlm"
eval_dataset_path = "./datasets/eval_dataset_13_03_2025_smolvlm"

train_dataset.save_to_disk(train_dataset_path)
eval_dataset.save_to_disk(eval_dataset_path)

print(f"Train dataset saved to: {train_dataset_path}")
print(f"Eval dataset saved to: {eval_dataset_path}")

Saving the dataset (0/1 shards):   0%|          | 0/3585 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/897 [00:00<?, ? examples/s]

Train dataset saved to: ./datasets/train_dataset_13_03_2025_smolvlm
Eval dataset saved to: ./datasets/eval_dataset_13_03_2025_smolvlm


In [15]:
#Remove Model and Clean GPU

def clear_memory():
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'peft_model' in globals(): del globals()['peft_model']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory() #Clearing variable, model and GPU

GPU allocated memory: 0.00 GB
GPU reserved memory: 0.00 GB


Fine Tune Model

In [16]:
model_id = "HuggingFaceTB/SmolVLM-Instruct" #"HuggingFaceTB/SmolVLM-500M-Instruct"

In [17]:
# from accelerate import PartialState
# device_map = "DDP" #This would enable Distributed Data Parallel

# if device_map == "DDP":
#     device_string = PartialState().process_index
#     device_map={'':device_string}

In [18]:
# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
model = Idefics3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)
processor = AutoProcessor.from_pretrained(model_id)

In [19]:
# from accelerate import Accelerator

# accelerator = Accelerator()
# device = accelerator.device  # Get the correct device managed by accelerate
# print(f"Using device: {device}")  # Should print cuda:0 or cuda:1

In [20]:
# model.to("cuda:0") #Bound the model to GPU 0 (Similary bound to whichever GPU is free) (Run this if not using DDP)
# print(next(model.parameters()).device)

In [21]:
# from accelerate import Accelerator

# accelerator = Accelerator()
# device = accelerator.device  # Get device assigned by accelerate

# print(f"Using device: {device}")  # Should print cuda:0 or cuda:1

# # Move the model to the correct device automatically
# model = accelerator.prepare(model)


In [22]:
# import torch
# from accelerate import Accelerator

# accelerator = Accelerator()
# device = accelerator.device

# print(f"Using device: {device}")  # Should print "cuda"
# print(f"Current CUDA device index: {torch.cuda.current_device()}")  
# print(f"Device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")


In [23]:
# Configure LoRA
peft_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
    use_dora=True,
    init_lora_weights="gaussian"
)

# Apply PEFT model adaptation
peft_model = get_peft_model(model, peft_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

trainable params: 11,269,248 || all params: 2,257,542,128 || trainable%: 0.4992


In [24]:
# Configure training arguments using SFTConfig
training_args = SFTConfig(
    output_dir="smolvlm-instruct-13-03-2025",
    logging_dir="smolvlm-instruct-13-03-2025/logs",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=10,
    optim="adamw_torch_fused",
    bf16=True,
    push_to_hub=True,
    report_to="tensorboard",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    #gradient_checkpointing_kwargs={"use_reentrant": False}, #Enable this when using DDP as device_map
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

In [25]:
image_token_id = processor.tokenizer.additional_special_tokens_ids[
            processor.tokenizer.additional_special_tokens.index("<image>")]

def collate_fn(examples):
    texts = [processor.apply_chat_template(example, tokenize=False) for example in examples]

    image_inputs = []
    for example in examples:
      image = example[1]['content'][0]['image']
      if image.mode != 'RGB':
          image = image.convert('RGB')
      image_inputs.append([image])

    batch = processor(text=texts, images=image_inputs, return_tensors="pt", padding=True)
    
    # Move all tensors to the same device as the model
    #batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100  # Mask padding tokens in labels
    labels[labels == image_token_id] = -100  # Mask image token IDs in labels

    batch["labels"] = labels

    return batch

In [26]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    data_collator=collate_fn,
    peft_config=peft_config,
    tokenizer=processor.tokenizer,
)

C:\Users\hal-pg2425-5\AppData\Local\Temp\8\ipykernel_38024\2552207196.py:1: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [27]:
%load_ext tensorboard
%tensorboard --logdir smolvlm-instruct-13-03-2025/logs --port=6010

In [28]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
10,2.279300
20,2.228400
30,2.113400
40,1.975100
50,1.795300
60,1.590000
70,1.387400
80,1.161700
90,1.036500
100,0.974600


TrainOutput(global_step=224, training_loss=1.1592926947133881, metrics={'train_runtime': 7848.0051, 'train_samples_per_second': 0.457, 'train_steps_per_second': 0.029, 'total_flos': 6.264369056276736e+16, 'train_loss': 1.1592926947133881})

In [29]:
trainer.save_model(training_args.output_dir)

In [30]:
trainer.push_to_hub()

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/aadhibest/smolvlm-instruct-13-03-2025/commit/b570728a0b1aae09a8676813361c39c4e97d24f4', commit_message='End of training', commit_description='', oid='b570728a0b1aae09a8676813361c39c4e97d24f4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aadhibest/smolvlm-instruct-13-03-2025', endpoint='https://huggingface.co', repo_type='model', repo_id='aadhibest/smolvlm-instruct-13-03-2025'), pr_revision=None, pr_num=None)

In [31]:
clear_memory()

GPU allocated memory: 0.02 GB
GPU reserved memory: 0.16 GB


In [32]:
# def generate_text_from_sample(model, processor, sample, max_new_tokens=1024, device="cuda"):
#     # Prepare the text input by applying the chat template
#     text_input = processor.apply_chat_template(
#         sample[1:2],  # Use the sample without the system message
#         add_generation_prompt=True
#     )

#     image_inputs = []
#     image = sample[1]['content'][0]['image']
#     if image.mode != 'RGB':
#         image = image.convert('RGB')
#     image_inputs.append([image])

#     # Prepare the inputs for the model
#     model_inputs = processor(
#         #text=[text_input],
#         text=text_input,
#         images=image_inputs,
#         return_tensors="pt",
#     ).to(device)  # Move inputs to the specified device

#     # Generate text with the model
#     generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

#     # Trim the generated ids to remove the input ids
#     trimmed_generated_ids = [
#         out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
#     ]

#     # Decode the output text
#     output_text = processor.batch_decode(
#         trimmed_generated_ids,
#         skip_special_tokens=True,
#         clean_up_tokenization_spaces=False
#     )

#     return output_text[0]  # Return the first decoded output text

In [33]:
# eval_dataset_formatted[0]

In [34]:
# model = Idefics3ForConditionalGeneration.from_pretrained(
#     model_id,
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
# )

# processor = AutoProcessor.from_pretrained(model_id)

In [35]:
# adapter_path = "smolvlm-500m-instruct-04-02-2025"
# model.load_adapter(adapter_path)

In [36]:
# eval_dataset_formatted[7]

In [37]:
# train_dataset_formatted[12]

In [38]:
# train_dataset_formatted[100][1]['content'][1]['text']

In [39]:
# train_dataset_formatted[1084][1]['content'][0]['image']

In [40]:
# eval_dataset_formatted[751][1]['content'][0]['image']

In [41]:
# eval_dataset_formatted[174]

In [42]:
# query = "Describe the signal flow in the circuit."
# for i in range(0,len(train_dataset_formatted)):
#     if query == train_dataset_formatted[i][1]['content'][1]['text']:
#         print(f"path: {train_dataset[i]["image"]} ## index: {i} ")

In [43]:
# output = generate_text_from_sample(model, processor, eval_dataset_formatted[25])
# output

In [44]:
# output = generate_text_from_sample(model, processor, eval_dataset_formatted[751])
# output

In [45]:
# output = generate_text_from_sample(model, processor, train_dataset_formatted[209])
# output

In [46]:
# type(train_dataset_formatted[8])

In [47]:
# train_dataset_formatted[0][1:2]

In [48]:
# train_dataset_formatted[209][1]['content'][0]['image']